<a href="https://colab.research.google.com/github/chavezaltamirano-ui/Growth-Models-in-Comparative/blob/main/Growth_Models_in_Comparative_Perspective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Growth Models in Comparative Perspective

1. Preparación del entorno en Colab

1.1. Librerías a instalar/importar

1. Bloque de Productividad y Crecimiento (PWT)
Este es el bloque medular para tus variables de PIB real, TFP, acervo de capital y capital humano.
•	Fuente: Penn World Table (PWT), versión 10.0 o superior.
•	Dirección: www.ggdc.net/pwt.
•	Procedimiento: Debes descargar el archivo completo (usualmente en formato Excel o Stata). La ventaja de este bloque es que ya viene homogeneizado por Paridad de Poder Adquisitivo (PPP), lo que permite la comparación directa entre China y EE. UU. sin necesidad de deflactores adicionales externos.


In [6]:
import pandas as pd
import os

# Crear carpetas
os.makedirs('data_raw', exist_ok=True)
os.makedirs('data_clean', exist_ok=True)

# URL del archivo Stata de PWT 11.0
pwt_url_dta = 'https://dataverse.nl/api/access/datafile/554030'  # Stata file de PWT 11.0

# Leer archivo Stata
pwt_df = pd.read_stata(pwt_url_dta)

# Filtrar China y Estados Unidos, años 1990–2025
countries_names = ['China', 'United States']
pwt_china_us = pwt_df[pwt_df['country'].isin(countries_names)]
pwt_china_us = pwt_china_us[pwt_china_us['year'].between(1990, 2025)]

# Seleccionar variables del bloque de productividad y crecimiento
# Usaremos:
# - rgdpna: Real GDP at constant national prices
# - rkna: Capital stock at constant national prices
# - rtfpna: Total factor productivity, nivel (rtfpna = TFP at constant national prices)
# - hc: Human capital index
# - pop: Population
cols_pwt = ['country', 'year', 'rgdpna', 'rkna', 'rtfpna', 'hc', 'pop']

missing_cols = [c for c in cols_pwt if c not in pwt_china_us.columns]
if missing_cols:
    print("ADVERTENCIA: estas columnas no se encontraron en PWT:", missing_cols)
else:
    pwt_china_us = pwt_china_us[cols_pwt]

    # Normalizar identificadores de país a códigos cortos (CN, US)
    country_map = {
        'China': 'CN',
        'United States': 'US'
    }
    pwt_china_us['country'] = pwt_china_us['country'].map(country_map)

    # Reordenar columnas: country, year, luego variables numéricas
    pwt_china_us = pwt_china_us[['country', 'year', 'rgdpna', 'rkna', 'rtfpna', 'hc', 'pop']]

    # Comprobación rápida
    print("\nVista previa del bloque PWT (China–US, 1990–2025):")
    print(pwt_china_us.head())

    # Guardar archivo intermedio y limpio
    raw_path = 'data_raw/pwt_china_us_1990_2025.csv'
    clean_path = 'data_clean/pwt_china_us_1990_2025.csv'

    pwt_china_us.to_csv(raw_path, index=False)
    pwt_china_us.to_csv(clean_path, index=False)

    print(f"\nArchivo guardado en: {raw_path}")
    print(f"Archivo también copiado en: {clean_path}")


Vista previa del bloque PWT (China–US, 1990–2025):
     country  year       rgdpna      rkna    rtfpna        hc          pop
2482      CN  1990  1857144.750  0.036908  0.351905  1.956077  1153.582724
2483      CN  1991  2029162.375  0.039276  0.366076  1.991197  1170.788528
2484      CN  1992  2317807.250  0.043118  0.396038  2.026947  1184.574237
2485      CN  1993  2639603.750  0.048899  0.421519  2.063339  1197.308575
2486      CN  1994  2983718.250  0.055067  0.446154  2.100384  1209.003096

Archivo guardado en: data_raw/pwt_china_us_1990_2025.csv
Archivo también copiado en: data_clean/pwt_china_us_1990_2025.csv


2. Bloque Financiero y de Endeudamiento (BIS, Banco Mundial e FMI)
Este bloque contiene las variables para identificar el modelo financiarizado (EE. UU.) y el financiamiento social total (China).
•	Fuentes clave:
o	BIS: Para estadísticas de crédito al sector no financiero y liquidez global (www.bis.org/statistics/totcredit.htm).
o	World Bank Databank: Para indicadores de desarrollo (WDI) y remesas (http://data.worldbank.org/).
o	IMF Datamapper: Para comparaciones rápidas de deuda y proyecciones (www.imf.org/external/datamapper).
•	Procedimiento: Aquí la descarga es por series de tiempo. Debes asegurar que las series cubran el periodo 1990-2025 para mantener la consistencia del dataset longitudinal.

In [22]:
# ============================================
# Bloque financiero WDI (World Bank)
# China (CN) y Estados Unidos (US), 1990–2025
# ============================================

import pandas as pd
import requests
import os

# 1. Crear carpetas para organizar los datos
os.makedirs('data_raw', exist_ok=True)
os.makedirs('data_clean', exist_ok=True)

# 2. Función auxiliar para descargar un indicador WDI vía API
def download_wdi_indicator(indicator_code, countries, start_year=1990, end_year=2025):
    """
    Descarga un indicador WDI para una lista de países y un rango de años.
    Usa la API del Banco Mundial:
    http://api.worldbank.org/v2/country/{country}/indicator/{indicator}
    Devuelve un DataFrame con columnas: country, year, indicator_code.
    """
    frames = []
    for country in countries:
        url = (
            f"http://api.worldbank.org/v2/country/{country}/indicator/{indicator_code}"
            f"?date={start_year}:{end_year}&format=json&per_page=20000"
        )
        resp = requests.get(url)
        if resp.status_code != 200:
            print(f"Error HTTP para {indicator_code}, país {country}: {resp.status_code}")
            continue

        data = resp.json()
        # data[1] contiene la lista de observaciones si la respuesta es válida
        if len(data) < 2 or data[1] is None:
            print(f"Sin datos para {indicator_code}, país {country}")
            continue

        rows = []
        for entry in data[1]:
            year = entry.get('date')
            value = entry.get('value')
            # Convertimos año a entero cuando sea posible
            try:
                year_int = int(year)
            except (TypeError, ValueError):
                continue
            rows.append({'country': country, 'year': year_int, indicator_code: value})

        if rows:
            frames.append(pd.DataFrame(rows))

    if frames:
        df = pd.concat(frames, ignore_index=True)
        return df
    else:
        # DataFrame vacío con columnas estándar
        return pd.DataFrame(columns=['country', 'year', indicator_code])

# 3. Definir países y rango de años
countries = ['CN', 'US']
start_year, end_year = 1990, 2025

# 4. Indicadores financieros WDI a descargar
# FS.AST.PRVT.GD.ZS: Domestic credit to private sector (% of GDP)
# FS.AST.DOMS.GD.ZS: Domestic credit provided by financial sector (% of GDP)
# BX.TRF.PWKR.DT.GD.ZS: Personal remittances, received (% of GDP)
indicators_financial = [
    'FS.AST.PRVT.GD.ZS',
    'FS.AST.DOMS.GD.ZS',
    'BX.TRF.PWKR.DT.GD.ZS'
]

# 5. Descargar cada indicador y fusionar en un solo DataFrame
df_fin = None

for ind in indicators_financial:
    print(f"Descargando indicador: {ind}")
    df_ind = download_wdi_indicator(ind, countries, start_year, end_year)
    if df_fin is None:
        df_fin = df_ind
    else:
        df_fin = df_fin.merge(df_ind, on=['country', 'year'], how='outer')

# 6. Renombrar columnas a nombres internos más claros
rename_map = {
    'FS.AST.PRVT.GD.ZS': 'credit_priv_gdp',    # crédito al sector privado (% del PIB)
    'FS.AST.DOMS.GD.ZS': 'credit_finsec_gdp', # crédito del sector financiero (% del PIB)
    'BX.TRF.PWKR.DT.GD.ZS': 'remittances_gdp' # remesas recibidas (% del PIB)
}
df_fin = df_fin.rename(columns=rename_map)

# 7. Ordenar por país y año
df_fin = df_fin.sort_values(['country', 'year'])

# 8. Comprobación rápida
print("\nVista previa del bloque financiero WDI:")
print(df_fin.head())

# 9. Guardar archivos en data_raw y data_clean
raw_path = 'data_raw/wdi_fin_china_us_1990_2025.csv'
clean_path = 'data_clean/wdi_fin_china_us_1990_2025.csv'

df_fin.to_csv(raw_path, index=False)
df_fin.to_csv(clean_path, index=False)

print(f"\nArchivo guardado en: {raw_path}")
print(f"Archivo también copiado en: {clean_path}")

Descargando indicador: FS.AST.PRVT.GD.ZS
Descargando indicador: FS.AST.DOMS.GD.ZS
Descargando indicador: BX.TRF.PWKR.DT.GD.ZS


/tmp/ipykernel_1460/428483796.py:54: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(frames, ignore_index=True)



Vista previa del bloque financiero WDI:
  country  year  credit_priv_gdp  credit_finsec_gdp  remittances_gdp
0      CN  1990        86.033020                NaN         0.054193
1      CN  1991        88.101074                NaN         0.101381
2      CN  1992        86.053063                NaN         0.144545
3      CN  1993        96.506392                NaN         0.141388
4      CN  1994        85.487555                NaN         0.151165

Archivo guardado en: data_raw/wdi_fin_china_us_1990_2025.csv
Archivo también copiado en: data_clean/wdi_fin_china_us_1990_2025.csv


In [24]:
# ============================================
# Exploración de archivos en Colab
# Directorio actual, data_raw y data_clean
# ============================================

import os

# 1. Archivos en el directorio actual
print("=== Archivos en el directorio actual ===")
if os.path.isdir("."):
    for f in os.listdir("."):
        print(f)
else:
    print("El directorio actual no existe (algo raro).")

# 2. Archivos en data_raw
print("\n=== Archivos en data_raw ===")
if os.path.isdir("data_raw"):
    for f in os.listdir("data_raw"):
        print(f)
else:
    print("La carpeta data_raw no existe.")

# 3. Archivos en data_clean
print("\n=== Archivos en data_clean ===")
if os.path.isdir("data_clean"):
    for f in os.listdir("data_clean"):
        print(f)
else:
    print("La carpeta data_clean no existe.")

# 4. Resumen: CSV y Excel en estos directorios
print("\n=== Archivos CSV y Excel en el directorio actual ===")
if os.path.isdir("."):
    for f in os.listdir("."):
        if f.lower().endswith((".csv", ".xlsx", ".xls")):
            print(f)

print("\n=== Archivos CSV y Excel en data_raw ===")
if os.path.isdir("data_raw"):
    for f in os.listdir("data_raw"):
        if f.lower().endswith((".csv", ".xlsx", ".xls")):
            print(f)

print("\n=== Archivos CSV y Excel en data_clean ===")
if os.path.isdir("data_clean"):
    for f in os.listdir("data_clean"):
        if f.lower().endswith((".csv", ".xlsx", ".xls")):
            print(f)

=== Archivos en el directorio actual ===
.config
data_raw
data_clean
sample_data

=== Archivos en data_raw ===
pwt_china_us_1990_2025.csv
wdi_fin_china_us_1990_2025.csv
bis_total_credit_csv.zip
totcredit.xlsx

=== Archivos en data_clean ===
pwt_china_us_1990_2025.csv
wdi_fin_china_us_1990_2025.csv

=== Archivos CSV y Excel en el directorio actual ===

=== Archivos CSV y Excel en data_raw ===
pwt_china_us_1990_2025.csv
wdi_fin_china_us_1990_2025.csv
totcredit.xlsx

=== Archivos CSV y Excel en data_clean ===
pwt_china_us_1990_2025.csv
wdi_fin_china_us_1990_2025.csv


In [25]:
# ============================================
# Bloque BIS mínimo - extracción desde totcredit.xlsx
# China (CN) y Estados Unidos (US)
# ============================================

import pandas as pd
import os

# 1. Rutas de archivos
bis_xlsx_path = "data_raw/totcredit.xlsx"

if not os.path.exists(bis_xlsx_path):
    raise FileNotFoundError("No se encontró data_raw/totcredit.xlsx")

# 2. Leer la hoja 'Quarterly Series' tal como está
xls = pd.ExcelFile(bis_xlsx_path)
print("Hojas disponibles en totcredit.xlsx:", xls.sheet_names)

sheet_q = "Quarterly Series"
df_q = pd.read_excel(bis_xlsx_path, sheet_name=sheet_q, header=None)

print("\nDimensiones de 'Quarterly Series':", df_q.shape)
print("Primeras filas (para referencia):")
print(df_q.iloc[:5, :10])

# 3. Construir una tabla de metadatos de columnas

meta_list = []

for j in range(1, df_q.shape[1]):
    header = df_q.iloc[0, j]   # texto largo de la serie
    unit   = df_q.iloc[1, j]   # unidad (Per cent, USD, etc.)
    area   = df_q.iloc[2, j]   # área o país (o agregado)
    code   = df_q.iloc[3, j]   # código BIS
    meta_list.append({
        "col_index": j,
        "header": str(header),
        "unit": str(unit),
        "area": str(area),
        "code": str(code),
    })

meta = pd.DataFrame(meta_list)

print("\nEjemplo de metadatos de columnas:")
print(meta.head(20))

# 4. Filtrar metadatos que parecen corresponder a China y Estados Unidos y % del PIB

mask_china = meta["header"].str.contains("China", case=False, na=False)
mask_us    = meta["header"].str.contains("United States", case=False, na=False)
mask_pct   = meta["header"].str.contains("Percentage of GDP", case=False, na=False)

meta_china_pct = meta[mask_china & mask_pct]
meta_us_pct    = meta[mask_us & mask_pct]

print("\nColumnas BIS para China (% del PIB):")
print(meta_china_pct)

print("\nColumnas BIS para United States (% del PIB):")
print(meta_us_pct)

# 5. Extraer las columnas seleccionadas y la columna de fechas

# La columna 0 contiene fechas y etiquetas; identificamos filas que parecen fechas.
date_raw = df_q.iloc[:, 0]

mask_date = ~date_raw.isna() & ~date_raw.astype(str).str.contains("Back to menu", case=False, na=False) & \
            ~date_raw.astype(str).str.contains("Period", case=False, na=False)

date_clean = date_raw[mask_date]
dates = pd.to_datetime(date_clean, errors="coerce")

df_bis_extracted = pd.DataFrame({"date": dates})
valid_idx = date_clean.index

# Añadir columnas de China
for _, row in meta_china_pct.iterrows():
    j = row["col_index"]
    col_name = f"CN_{row['code']}"
    values = df_q.iloc[valid_idx, j].reset_index(drop=True)
    df_bis_extracted[col_name] = values

# Añadir columnas de Estados Unidos
for _, row in meta_us_pct.iterrows():
    j = row["col_index"]
    col_name = f"US_{row['code']}"
    values = df_q.iloc[valid_idx, j].reset_index(drop=True)
    df_bis_extracted[col_name] = values

print("\nVista previa de df_bis_extracted (trimestral):")
print(df_bis_extracted.head())

# 6. Guardar este extracto como base BIS cruda

os.makedirs("data_raw", exist_ok=True)
os.makedirs("data_clean", exist_ok=True)

raw_path_bis   = "data_raw/bis_totcredit_extract_cn_us.csv"
clean_path_bis = "data_clean/bis_totcredit_extract_cn_us.csv"

df_bis_extracted.to_csv(raw_path_bis, index=False)
df_bis_extracted.to_csv(clean_path_bis, index=False)

print(f"\nArchivo BIS EXTRAÍDO guardado en: {raw_path_bis}")
print(f"Archivo BIS EXTRAÍDO también copiado en: {clean_path_bis}")

Hojas disponibles en totcredit.xlsx: ['Content', 'Summary Documentation', 'Quarterly Series']

Dimensiones de 'Quarterly Series': (337, 1134)
Primeras filas (para referencia):
                     0                                                  1  \
0         Back to menu  Emerging market economies (aggregate) - Credit...   
1                  NaN                                   Per cent (Units)   
2                  NaN              Emerging market economies (aggregate)   
3               Period                                   Q:4T:C:A:M:770:A   
4  1940-06-30 00:00:00                                                NaN   

                                                   2  \
0  Emerging market economies (aggregate) - Credit...   
1                                   Per cent (Units)   
2              Emerging market economies (aggregate)   
3                                   Q:4T:C:A:M:799:A   
4                                                NaN   

                        

In [27]:
# ============================================
# Bloque BIS anual - crédito al sector no financiero
# Usa bis_totcredit_extract_cn_us.csv (trimestral)
# Para China (CN) y Estados Unidos (US), 1990–2025
# ============================================

import pandas as pd
import os

# 1. Leer el extracto BIS trimestral
extract_path = "data_clean/bis_totcredit_extract_cn_us.csv"
if not os.path.exists(extract_path):
    extract_path = "data_raw/bis_totcredit_extract_cn_us.csv"

if not os.path.exists(extract_path):
    raise FileNotFoundError("No se encontró bis_totcredit_extract_cn_us.csv en data_raw/data_clean.")

df_q = pd.read_csv(extract_path)

print("Columnas del extracto BIS trimestral:")
print(df_q.columns)

print("\nVista rápida del extracto:")
print(df_q.head())

# 2. Asegurar que la columna date es de tipo datetime y crear año
df_q["date"] = pd.to_datetime(df_q["date"], errors="coerce")
df_q = df_q.dropna(subset=["date"])

df_q["year"] = df_q["date"].dt.year

# 3. Filtrar rango de años 1990–2025 (ajusta si necesitas otro rango)
df_q = df_q[(df_q["year"] >= 1990) & (df_q["year"] <= 2025)]

# 4. Definir mapping de columnas -> país + nombre interno de serie

col_map = {
    # China
    "CN_Q:CN:C:A:M:770:A": ("CN", "bis_tot_credit_gdp"),     # total crédito sector no financiero
    "CN_Q:CN:P:A:M:770:A": ("CN", "bis_pvt_credit_gdp"),     # crédito privado no financiero (todos acreedores)
    "CN_Q:CN:H:A:M:770:A": ("CN", "bis_hh_credit_gdp"),      # hogares y NPISHs
    "CN_Q:CN:N:A:M:770:A": ("CN", "bis_nfc_credit_gdp"),     # corporaciones no financieras
    "CN_Q:CN:G:A:N:770:A": ("CN", "bis_gov_credit_gdp"),     # gobierno (nominal, % PIB)
    "CN_Q:CN:P:B:M:770:A": ("CN", "bis_pvt_banks_credit_gdp"),  # crédito privado desde bancos (opcional)

    # Estados Unidos
    "US_Q:US:C:A:M:770:A": ("US", "bis_tot_credit_gdp"),
    "US_Q:US:P:A:M:770:A": ("US", "bis_pvt_credit_gdp"),
    "US_Q:US:H:A:M:770:A": ("US", "bis_hh_credit_gdp"),
    "US_Q:US:N:A:M:770:A": ("US", "bis_nfc_credit_gdp"),
    # Para gobierno en US tienes dos series: G:A:M:770:A y G:A:N:770:A.
    # Aquí usamos la nominal (N) para ser consistentes con China:
    "US_Q:US:G:A:N:770:A": ("US", "bis_gov_credit_gdp"),
    "US_Q:US:P:B:M:770:A": ("US", "bis_pvt_banks_credit_gdp"),
}

# 5. Construir DataFrame largo con columnas estandarizadas:
# country, year, series_short, value

records = []

for col in df_q.columns:
    if col == "date" or col == "year":
        continue
    if col not in col_map:
        continue

    country, series_short = col_map[col]
    series_values = df_q[col].values
    dates = df_q["date"].values
    years = df_q["year"].values

    for dt, yr, val in zip(dates, years, series_values):
        records.append({
            "country": country,
            "year": yr,
            "date": dt,
            "series_short": series_short,
            "value": val,
        })

df_long = pd.DataFrame(records)

print("\nVista rápida de df_long (BIS largo trimestral):")
print(df_long.head())

# 6. Pasar de trimestral a anual: quedarnos con el último trimestre de cada año

df_long = (
    df_long
    .sort_values(["country", "series_short", "year", "date"])
    .groupby(["country", "series_short", "year"], as_index=False)
    .tail(1)
)

print("\nVista tras agregación anual (último trimestre):")
print(df_long.head())

# 7. Pivotear a formato ancho por país y año

df_bis_annual = (
    df_long
    .pivot(index=["country", "year"], columns="series_short", values="value")
    .reset_index()
)

df_bis_annual = df_bis_annual.sort_values(["country", "year"])

print("\nVista previa de df_bis_annual (BIS anual crédito % PIB):")
print(df_bis_annual.head())

# 8. Guardar en data_raw y data_clean

os.makedirs("data_raw", exist_ok=True)
os.makedirs("data_clean", exist_ok=True)

raw_path_bis   = "data_raw/bis_totcredit_cn_us_1990_2025.csv"
clean_path_bis = "data_clean/bis_totcredit_cn_us_1990_2025.csv"

df_bis_annual.to_csv(raw_path_bis, index=False)
df_bis_annual.to_csv(clean_path_bis, index=False)

print(f"\nArchivo BIS ANUAL guardado en: {raw_path_bis}")
print(f"Archivo BIS ANUAL también copiado en: {clean_path_bis}")

Columnas del extracto BIS trimestral:
Index(['date', 'CN_Q:CN:C:A:M:770:A', 'CN_Q:CN:G:A:N:770:A',
       'CN_Q:CN:H:A:M:770:A', 'CN_Q:CN:N:A:M:770:A', 'CN_Q:CN:P:A:M:770:A',
       'CN_Q:CN:P:B:M:770:A', 'US_Q:US:C:A:M:770:A', 'US_Q:US:G:A:M:770:A',
       'US_Q:US:G:A:N:770:A', 'US_Q:US:H:A:M:770:A', 'US_Q:US:N:A:M:770:A',
       'US_Q:US:P:A:M:770:A', 'US_Q:US:P:B:M:770:A'],
      dtype='object')

Vista rápida del extracto:
         date  CN_Q:CN:C:A:M:770:A  CN_Q:CN:G:A:N:770:A  CN_Q:CN:H:A:M:770:A  \
0  1940-06-30                  NaN                  NaN                  NaN   
1  1940-09-30                  NaN                  NaN                  NaN   
2  1940-12-31                  NaN                  NaN                  NaN   
3  1941-03-31                  NaN                  NaN                  NaN   
4  1941-06-30                  NaN                  NaN                  NaN   

   CN_Q:CN:N:A:M:770:A  CN_Q:CN:P:A:M:770:A  CN_Q:CN:P:B:M:770:A  \
0                  N